In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

# Ensure timestamp numeric
for df in (metric, trace, log, error_logs):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

# Helper to get min/max timestamps as timezone-aware datetimes (UTC+8)
def ts_min_max(df):
    ts = pd.to_datetime(df['timestamp'].dropna().astype(int), unit='s', utc=True)
    if ts.empty:
        return (pd.NaT, pd.NaT)
    ts = ts.dt.tz_convert(tz)
    return (ts.min(), ts.max())

# Metric summaries
metric_ts_min, metric_ts_max = ts_min_max(metric)
metric_meta = pd.DataFrame({
    'rows': [len(metric)],
    'ts_min': [metric_ts_min],
    'ts_max': [metric_ts_max]
})
metric_top_cmdb = metric['cmdb_id'].value_counts().head(50).rename_axis('cmdb_id').reset_index(name='count')
metric_top_kpis = metric['kpi_name'].value_counts().head(20).rename_axis('kpi_name').reset_index(name='count')

# Trace summaries
trace_ts_min, trace_ts_max = ts_min_max(trace)
trace_meta = pd.DataFrame({
    'rows': [len(trace)],
    'ts_min': [trace_ts_min],
    'ts_max': [trace_ts_max]
})
trace_top_cmdb = trace['cmdb_id'].value_counts().head(50).rename_axis('cmdb_id').reset_index(name='count')
trace_top_names = trace['trace_name'].value_counts().head(20).rename_axis('trace_name').reset_index(name='count')

# Log summaries
log_ts_min, log_ts_max = ts_min_max(log)
log_meta = pd.DataFrame({
    'rows': [len(log)],
    'ts_min': [log_ts_min],
    'ts_max': [log_ts_max]
})
log_top_cmdb = log['cmdb_id'].value_counts().head(50).rename_axis('cmdb_id').reset_index(name='count')
log_top_names = log['log_name'].value_counts().head(20).rename_axis('log_name').reset_index(name='count')

# Error_logs summaries
err_ts_min, err_ts_max = ts_min_max(error_logs)
error_meta = pd.DataFrame({
    'rows': [len(error_logs)],
    'ts_min': [err_ts_min],
    'ts_max': [err_ts_max]
})
error_top_messages = error_logs['message'].value_counts().head(20).rename_axis('message').reset_index(name='count')
error_earliest5 = error_logs.sort_values('timestamp').head(5).copy()
# convert earliest timestamps for display
if not error_earliest5.empty:
    error_earliest5['ts_local'] = pd.to_datetime(error_earliest5['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
    error_earliest5 = error_earliest5[['ts_local','cmdb_id','message']]

# Display compact summaries (limited rows where applicable)
metric_meta, metric_top_cmdb.head(50), metric_top_kpis.head(20), trace_meta, trace_top_cmdb.head(50), trace_top_names.head(20), log_meta, log_top_cmdb.head(50), log_top_names.head(20), error_meta, error_top_messages.head(20), error_earliest5.head(5)

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry files:\n\n"
    "metric.csv: 145,812 rows; timestamps 2022-03-21 01:30 -> 2022-03-21 01:59 (UTC+8).\n"
    "  Top cmdb_ids (by rows, top 5): adservice (10230), adservice2 (9990), frontend2-0 (5040), frontend-0 (4800), frontend-1 (4680).\n"
    "  Top KPI themes: many container.node-5.* metrics (network receive/transmit, errors, memory metrics) with top counts ~1050 and ~990.\n\n"
    "trace.csv: 28,944 rows; timestamps 2022-03-21 01:30 -> 2022-03-21 01:59 (UTC+8).\n"
    "  Top cmdb_ids (by rows, top 5): frontend-0 (2584), frontend-2 (2504), frontend-1 (2496), checkoutservice-2 (1480), checkoutservice-1 (1464).\n"
    "  Top trace names (top examples): trace.from_frontend-* duration_mean / duration_p95 / error_rate / row_count (counts 564–586 for frontend pods).\n\n"
    "log.csv: 1,720 rows; timestamps 2022-03-21 01:30 -> 2022-03-21 01:59 (UTC+8).\n"
    "  Top cmdb_ids include adservice-0/1/2 and various service pods.\n"
    "  Top log names: log.error_count (860), log.row_count (860).\n\n"
    "error_logs.csv: 543 rows; timestamps 2022-03-21 01:30:04 -> 2022-03-21 01:59:59 (UTC+8).\n"
    "  Top message (all 543): 'severity: warning, message: failed to retrieve...'.\n"
    "  Earliest 5 error rows: 2022-03-21 01:30:04 -> frontend-0, frontend-1; 01:30:06 -> frontend-2; 01:30:11 -> frontend-1; 01:30:12 -> frontend-2 (all same message).\n\n"
    "Quick interpretation: error logs are concentrated in frontend pods (identical warning message across all 543 entries) and traces show frontend-related latency/error series. Metric KPIs show heavy activity in node-5 network/memory metrics. Immediate investigation candidates: frontend pods (frontend-0/1/2) and node-5 (network/memory metrics).\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     rows                    ts_min                    ts_max
0  145812 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00,           cmdb_id  count
0       adservice  10230
1      adservice2   9990
2     frontend2-0   5040
3      frontend-0   4800
4      frontend-1   4680
..            ...    ...
45         node-1   1692
46         node-3   1692
47         node-5   1692
48  redis-cart2-0    360
49   redis-cart-0    360

[50 rows x 2 columns],                                              kpi_name  count
0   container.node-5.container_network_receive_MB....   1050
1   container.node-5.container_network_receive_pac...   1050
2   container.node-5.container_network_transmit_pa...   1050
3   container.node-5.container_network_receive_err...   1050
4   container.node-5.container_network_transmit_er...   1050
5   container.node-5.container_network_transmit_MB...   1050
6   container.node-5.container_network_receive_pac...   1050
7   container.node-5.container_network_transmit_pa...   1050
8   container.node-5.container_memory_failures.con...    990
9               container.node-5.container_memory_rss    990
10  container.node-5.container_memory_failures.hie...    990
11             container.node-5.container_memory_swap    990
12   container.node-5.container_tasks_state.iowaiting    990
13  container.node-5.container_spec_memory_swap_li...    990
14  container.node-5.container_fs_writes_merged./d...    990
15         container.node-5.container_spec_cpu_period    990
16             container.node-5.container_threads_max    990
17  container.node-5.container_ulimits_soft.max_op...    990
18     container.node-5.container_tasks_state.stopped    990
19    container.node-5.container_tasks_state.sleeping    990,     rows                    ts_min                    ts_max
0  28944 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00,               cmdb_id  count
0          frontend-0   2584
1          frontend-2   2504
2          frontend-1   2496
3   checkoutservice-2   1480
4   checkoutservice-1   1464
..                ...    ...
35     emailservice-1    136
36     emailservice-2    128
37       adservice2-0    120
38    emailservice2-0     96
39  paymentservice2-0     96

[40 rows x 2 columns],                                     trace_name  count
0          trace.from_frontend-0.duration_mean    586
1           trace.from_frontend-0.duration_p95    586
2             trace.from_frontend-0.error_rate    586
3              trace.from_frontend-0.row_count    586
4              trace.from_frontend-2.row_count    566
5             trace.from_frontend-2.error_rate    566
6           trace.from_frontend-2.duration_p95    566
7          trace.from_frontend-2.duration_mean    566
8              trace.from_frontend-1.row_count    564
9             trace.from_frontend-1.error_rate    564
10          trace.from_frontend-1.duration_p95    564
11         trace.from_frontend-1.duration_mean    564
12                        trace.self.row_count    448
13                       trace.self.error_rate    448
14                     trace.self.duration_p95    448
15                    trace.self.duration_mean    448
16  trace.from_checkoutservice-2.duration_mean    292
17   trace.from_checkoutservice-2.duration_p95    292
18     trace.from_checkoutservice-2.error_rate    292
19      trace.from_checkoutservice-2.row_count    292,    rows                    ts_min                    ts_max
0  1720 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00,               cmdb_id  count
0         adservice-0     60
1         adservice-1     60
2         adservice-2     60
3       cartservice-0     60
4       cartservice-1     60
..                ...    ...
26     emailservice-2     48
27  checkoutservice-2     48
28  checkoutservice-1     46
29  checkoutservice-0     44
30       redis-cart-0      8

[31 rows x 2 columns],           log_name  count
0  log.error_count    860
1    log.row_count    860,    rows                    ts_min                    ts_max
0   543 2022-03-21 01:30:04+08:00 2022-03-21 01:59:59+08:00,                                              message  count
0  severity: warning, message: failed to retrieve...    543,                    ts_local     cmdb_id                                            message
0 2022-03-21 01:30:04+08:00  frontend-0  severity: warning, message: failed to retrieve...
1 2022-03-21 01:30:04+08:00  frontend-1  severity: warning, message: failed to retrieve...
2 2022-03-21 01:30:06+08:00  frontend-2  severity: warning, message: failed to retrieve...
3 2022-03-21 01:30:11+08:00  frontend-1  severity: warning, message: failed to retrieve...
4 2022-03-21 01:30:12+08:00  frontend-2  severity: warning, message: failed to retrieve...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse existing 'metric' if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure timestamp numeric and value numeric
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Work on a compact copy
m = metric[['timestamp', 'cmdb_id', 'kpi_name', 'value']].copy()

# Group and compute required aggregations (global across full file)
grp = m.groupby(['cmdb_id', 'kpi_name'], sort=False)
agg = grp.agg(
    count=('value', 'size'),
    p95_value=('value', lambda x: float(x.quantile(0.95)))  # global P95 per component-KPI
).reset_index()

# Compute ts_min and ts_max (as ints) then convert to UTC+8 datetimes for display
ts_bounds = grp.agg(ts_min=('timestamp', 'min'), ts_max=('timestamp', 'max')).reset_index()
# Merge
agg = agg.merge(ts_bounds, on=['cmdb_id', 'kpi_name'], how='left')

# Convert ts_min/ts_max to timezone-aware datetimes in UTC+8 for readability
def to_local_series(ts_series):
    # handle missing values
    s = pd.to_datetime(ts_series.dropna().astype(int), unit='s', utc=True).dt.tz_convert(tz)
    # map back preserving index
    result = pd.Series(index=ts_series.index, dtype='datetime64[ns, Asia/Shanghai]')
    for idx, val in s.items():
        result.at[idx] = val
    return result

agg['ts_min'] = to_local_series(agg['ts_min'])
agg['ts_max'] = to_local_series(agg['ts_max'])

# Filter to requested cmdb_ids and sort by count desc, limit to top 20
target_cmdb = ["frontend-0","frontend-1","frontend-2","node-5"]
result_filtered = (
    agg[agg['cmdb_id'].isin(target_cmdb)]
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'count', 'ts_min', 'ts_max', 'p95_value']]
    .head(20)
)

result_filtered

```
Out[2]:
```
```python
# Summarize the previously computed filtered aggregation (result_filtered)
try:
    rf = result_filtered
except NameError:
    raise RuntimeError("result_filtered not found in kernel state. Please run the aggregation cell first.")

n_rows = len(rf)
cmdb_list = ", ".join(sorted(rf['cmdb_id'].unique()))
counts_summary = rf['count'].value_counts().to_dict()
ts_min = rf['ts_min'].min()
ts_max = rf['ts_max'].max()

# prepare up to 5 example rows (compact)
examples = []
for _, r in rf.head(5).iterrows():
    examples.append(f"{r['cmdb_id']} | {r['kpi_name']} | count={int(r['count'])} | p95={r['p95_value']:.6g}")

summary = (
    f"Aggregated metric summary (filtered to frontend-0, frontend-1, frontend-2, node-5):\n\n"
    f"Returned rows: {n_rows} (top 20 by count).\n"
    f"Components present: {cmdb_list}.\n"
    f"Counts distribution (value -> number of component-KPIs): {counts_summary}.\n"
    f"Timestamp range across these component-KPIs: {ts_min} -> {ts_max} (UTC+8).\n\n"
    f"Examples (cmdb_id | kpi_name | count | p95_value):\n  - " + "\n  - ".join(examples) + "\n\n"
    f"Notes: Most returned component-KPI groups have count=30 (30 data points) covering 2022-03-21 01:30 -> 01:59 UTC+8. "
    f"Node-5 KPIs include system IO/load/disk metrics (e.g., system.udp.connect.num p95=9.0, system.disk.used p95≈4.463e7). "
    f"Frontend-0 rows correspond to container.* KPIs (examples include cpu_usage_seconds p95≈0.29846, file_descriptors p95=18).\n"
    f"If you want the full filtered table, request to display result_filtered.head(X) or the entire result_filtered."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count                    ts_min                    ts_max     p95_value
3405      node-5                             system.udp.connect.num     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  9.000000e+00
2490  frontend-0         container.node-5.container_cpu_cfs_periods     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  1.093750e+02
2491  frontend-0  container.node-5.container_cpu_cfs_throttled_p...     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  0.000000e+00
2492  frontend-0  container.node-5.container_cpu_cfs_throttled_s...     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  0.000000e+00
2493  frontend-0    container.node-5.container_cpu_load_average_10s     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  0.000000e+00
2494  frontend-0      container.node-5.container_cpu_system_seconds     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  9.500000e-02
2495  frontend-0       container.node-5.container_cpu_usage_seconds     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  2.984616e-01
2496  frontend-0        container.node-5.container_cpu_user_seconds     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  1.777500e-01
2497  frontend-0        container.node-5.container_file_descriptors     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  1.800000e+01
2498  frontend-0     container.node-5.container_fs_inodes./dev/vda1     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  0.000000e+00
3366      node-5                                      system.io.r_s     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  0.000000e+00
3367      node-5                                    system.io.rkb_s     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  0.000000e+00
3368      node-5                                    system.io.svctm     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  2.305000e-01
3369      node-5                                     system.io.util     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  8.275000e-01
3370      node-5                                  system.io.w_await     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  2.551000e+00
3371      node-5                                      system.io.w_s     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  5.200000e+01
3372      node-5                                      system.load.1     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  4.255000e-01
3373      node-5                                     system.load.15     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  5.310000e-01
3358      node-5                                   system.disk.used     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  4.463259e+07
3359      node-5                              system.fs.inodes.free     30 2022-03-21 01:30:00+08:00 2022-03-21 01:59:00+08:00  3.899833e+07```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse variables from kernel state
try:
    metric
    result_filtered
    tz
except NameError:
    # load if missing (defensive)
    metric = pd.read_csv('metric.csv')
    result_filtered = pd.read_csv('result_filtered.csv')  # unlikely; kept for safety
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric types
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Prepare local timestamp (UTC epoch -> UTC aware -> convert to UTC+8)
metric['ts_local'] = pd.to_datetime(metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Define window in UTC+8 (inclusive)
window_start = pd.Timestamp("2022-03-21 01:30:00", tz=tz)
window_end = pd.Timestamp("2022-03-21 01:59:59", tz=tz)

# Target cmdbs
targets = ["frontend-0","frontend-1","frontend-2","node-5"]

# Filter metric rows by cmdb and time window
m_window = metric[
    metric['cmdb_id'].isin(targets) &
    (metric['ts_local'] >= window_start) &
    (metric['ts_local'] <= window_end)
].copy()

# Prepare p95 lookup from previously computed global thresholds (result_filtered)
# result_filtered should include columns ['cmdb_id','kpi_name','p95_value']
p95_lookup = result_filtered[['cmdb_id','kpi_name','p95_value']].drop_duplicates()

# Join metric rows to p95 thresholds (inner join ensures only groups with precomputed p95 are considered)
m_join = m_window.merge(p95_lookup, on=['cmdb_id','kpi_name'], how='inner')

# Define anomaly flag
m_join['is_anom'] = m_join['value'] > m_join['p95_value']

# Aggregate per component-KPI
grouped = m_join.groupby(['cmdb_id','kpi_name'], sort=False)

agg_basic = grouped.agg(
    anomaly_count = ('is_anom', 'sum'),
    max_value_in_window = ('value', 'max'),
    p95_value = ('p95_value', 'first')
).reset_index()

# earliest anomaly timestamp (for groups with anomalies)
anoms = m_join[m_join['is_anom']]
if not anoms.empty:
    earliest = anoms.groupby(['cmdb_id','kpi_name'], sort=False)['ts_local'].min().reset_index().rename(columns={'ts_local':'earliest_anomaly_timestamp'})
    agg = agg_basic.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
else:
    agg = agg_basic.copy()
    agg['earliest_anomaly_timestamp'] = pd.NaT

# Calculate breach_ratio safely (handle p95_value == 0)
agg['breach_ratio'] = np.where(
    agg['p95_value'] == 0,
    np.where(agg['max_value_in_window'] > 0, np.inf, 0.0),
    agg['max_value_in_window'] / agg['p95_value']
)

# Keep only groups with at least one anomaly
anomalies_summary = agg[agg['anomaly_count'] > 0].copy()

# Format earliest_anomaly_timestamp as timezone-aware (already is) and keep requested columns
if not anomalies_summary.empty:
    anomalies_summary = anomalies_summary.loc[:, ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','breach_ratio']]
    # Sort by breach_ratio desc and limit to top 20
    anomalies_summary = anomalies_summary.sort_values('breach_ratio', ascending=False).head(20).reset_index(drop=True)
else:
    # per instruction: return an empty list indicator if no anomalies found
    anomalies_summary = []

# Display result (variable)
anomalies_summary

```
Out[3]:
```
```python
# Summarize anomalies found (compact, plain English) using the DataFrame in kernel: anomalies_summary
import pandas as pd
try:
    df = anomalies_summary
except NameError:
    df = pd.DataFrame()

if df.empty:
    summary = "No anomalies found in the specified window for frontend-0/1/2 and node-5."
else:
    total = len(df)
    by_component = df['cmdb_id'].value_counts().to_dict()
    top5 = df.sort_values('breach_ratio', ascending=False).head(5)
    top_lines = []
    for _, r in top5.iterrows():
        ts = r['earliest_anomaly_timestamp']
        ts_str = pd.to_datetime(ts).tz_convert('Asia/Shanghai').strftime('%Y-03-%d %H:%M:%S%z') if pd.notna(ts) else "N/A"
        top_lines.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | anomalies={int(r['anomaly_count'])} | earliest={ts_str} | "
            f"max={r['max_value_in_window']:.6g} | p95={r['p95_value']:.6g} | breach_ratio={r['breach_ratio']:.3f}"
        )
    summary = (
        f"Anomaly summary (window 2022-03-21 01:30:00 to 01:59:59 UTC+8):\n"
        f"- Total component-KPIs with anomalies: {total} (by component: {by_component}).\n"
        f"- Top 5 breaches by breach_ratio:\n  - " + "\n  - ".join(top_lines) + "\n\n"
        f"Interpretation: Most anomalies are on node-5 (IO / load metrics: system.io.svctm, system.load.*, system.io.*) with the largest breach_ratio ~3.08 for system.io.svctm. "
        f"Frontend-0 shows minor CPU-related breaches (breach_ratio ~1.05). Disk.used on node-5 is only marginally above p95 (breach_ratio ~1.00025).\n"
        f"Recommendation: Investigate node-5 I/O and load (svc time, write stalls, load average) as primary suspects; then review frontend CPU metrics for secondary effects."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window     p95_value  breach_ratio
0       node-5                                system.io.svctm              2  2022-03-21 01:47:00+08:00         7.100000e-01  2.305000e-01      3.080260
1       node-5                                  system.load.1              2  2022-03-21 01:40:00+08:00         8.200000e-01  4.255000e-01      1.927145
2       node-5                                  system.io.w_s              1  2022-03-21 01:42:00+08:00         6.900000e+01  5.200000e+01      1.326923
3       node-5                              system.io.w_await              2  2022-03-21 01:41:00+08:00         2.960000e+00  2.551000e+00      1.160329
4       node-5                                 system.io.util              2  2022-03-21 01:41:00+08:00         9.500000e-01  8.275000e-01      1.148036
5   frontend-0  container.node-5.container_cpu_system_seconds              1  2022-03-21 01:57:00+08:00         1.000000e-01  9.500000e-02      1.052632
6   frontend-0   container.node-5.container_cpu_usage_seconds              2  2022-03-21 01:42:00+08:00         3.127271e-01  2.984616e-01      1.047797
7   frontend-0     container.node-5.container_cpu_cfs_periods              2  2022-03-21 01:42:00+08:00         1.145000e+02  1.093750e+02      1.046857
8   frontend-0    container.node-5.container_cpu_user_seconds              2  2022-03-21 01:46:00+08:00         1.850000e-01  1.777500e-01      1.040788
9       node-5                                 system.load.15              2  2022-03-21 01:30:00+08:00         5.500000e-01  5.310000e-01      1.035782
10      node-5                               system.disk.used              2  2022-03-21 01:58:00+08:00         4.464390e+07  4.463259e+07      1.000253```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse existing trace and tz if present
try:
    trace
    tz
except NameError:
    trace = pd.read_csv('trace.csv')
    tz = pytz.timezone('Asia/Shanghai')

# Ensure types
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce').astype('Int64')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Compute global thresholds (across entire trace series) per (cmdb_id, trace_name)
grp = trace.groupby(['cmdb_id', 'trace_name'], sort=False)

p95 = grp['value'].quantile(0.95).reset_index(name='p95_value')
# compute p5 for all groups (we'll select for row_count later)
p5 = grp['value'].quantile(0.05).reset_index(name='p5_value')

# Merge thresholds
thresholds = p95.merge(p5, on=['cmdb_id','trace_name'], how='left')

# Decide threshold_value: use p5 for row_count groups, else p95
thresholds['is_row_count'] = thresholds['trace_name'].str.contains('row_count', case=False, na=False)
thresholds['threshold_value'] = np.where(thresholds['is_row_count'], thresholds['p5_value'], thresholds['p95_value'])
thresholds = thresholds[['cmdb_id','trace_name','threshold_value','is_row_count']]

# Define targets and trace_name patterns
targets = ["frontend-0","frontend-1","frontend-2"]
patterns = ['error_rate','duration_mean','duration_p95','row_count']

# Prepare local timestamp and filter window (UTC+8)
trace['ts_local'] = pd.to_datetime(trace['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)
window_start = pd.Timestamp("2022-03-21 01:30:00", tz=tz)
window_end = pd.Timestamp("2022-03-21 01:59:59", tz=tz)

# Filter trace rows by cmdb and name patterns and time window
mask_cmdb = trace['cmdb_id'].isin(targets)
mask_name = trace['trace_name'].str.contains('|'.join(patterns), case=False, na=False)
mask_time = (trace['ts_local'] >= window_start) & (trace['ts_local'] <= window_end)
t_window = trace[mask_cmdb & mask_name & mask_time].copy()

# Join with global thresholds (inner join to ensure groups have thresholds)
t_join = t_window.merge(thresholds, on=['cmdb_id','trace_name'], how='inner')

# Determine anomaly per row: for row_count anomalies when value < threshold; else value > threshold
t_join['is_row_count'] = t_join['is_row_count'].astype(bool)
t_join['is_anom'] = np.where(t_join['is_row_count'], t_join['value'] < t_join['threshold_value'], t_join['value'] > t_join['threshold_value'])

# Aggregate per (cmdb_id, trace_name)
agg = t_join.groupby(['cmdb_id','trace_name','is_row_count'], sort=False).agg(
    anomaly_count = ('is_anom', 'sum'),
    max_value_in_window = ('value', 'max'),
    min_value_in_window = ('value', 'min'),
    threshold_value = ('threshold_value', 'first')
).reset_index()

# earliest anomaly timestamp per group (only where anomalies exist)
earliest = t_join[t_join['is_anom']].groupby(['cmdb_id','trace_name'], sort=False)['ts_local'].min().reset_index(name='earliest_anomaly_timestamp')
agg = agg.merge(earliest, on=['cmdb_id','trace_name'], how='left')

# Compute extremal_value_in_window and breach_ratio
# For non-row_count (upper breach): extremal = max_value, breach_ratio = extremal / threshold
# For row_count (lower breach): extremal = min_value, breach_ratio = threshold / extremal
agg['extremal_value_in_window'] = np.where(agg['is_row_count'], agg['min_value_in_window'], agg['max_value_in_window'])

# handle division by zero and invalids
def compute_breach(row):
    thr = row['threshold_value']
    ext = row['extremal_value_in_window']
    if pd.isna(thr) or pd.isna(ext):
        return np.nan
    if row['is_row_count']:
        # threshold / ext (ext could be 0)
        if ext == 0:
            return np.inf if thr > 0 else np.nan
        return thr / ext
    else:
        # ext / threshold
        if thr == 0:
            return np.inf if ext > 0 else np.nan
        return ext / thr

agg['breach_ratio'] = agg.apply(compute_breach, axis=1)

# Keep only groups with at least one anomaly
anomaly_trace_summary = agg[agg['anomaly_count'] > 0].copy()

# Select and sort top by breach_ratio desc, limit 20, keep columns requested
if not anomaly_trace_summary.empty:
    anomaly_trace_summary = anomaly_trace_summary.loc[:, [
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
        'extremal_value_in_window','threshold_value','breach_ratio'
    ]].sort_values('breach_ratio', ascending=False).head(20).reset_index(drop=True)
else:
    anomaly_trace_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
        'extremal_value_in_window','threshold_value','breach_ratio'
    ])

# Display compact result
anomaly_trace_summary

```
Out[4]:
```
```python
# Summarize trace anomaly results (compact, plain English)
try:
    df = anomaly_trace_summary  # from previous cell
except NameError:
    df = pd.DataFrame()

if df.empty:
    summary = "No trace anomalies found for frontend-0/1/2 in the specified window."
else:
    total = len(df)
    components = ", ".join(sorted(df['cmdb_id'].unique()))
    # prepare top 5 by breach_ratio
    top5 = df.sort_values('breach_ratio', ascending=False).head(5)
    lines = []
    for _, r in top5.iterrows():
        br = r['breach_ratio']
        br_str = "inf" if (pd.isna(br) and r['threshold_value'] == 0) or (br == float('inf')) else f"{br:.3f}"
        ts = r['earliest_anomaly_timestamp']
        ts_str = ts.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(ts) else "N/A"
        lines.append(
            f"{r['cmdb_id']} | {r['trace_name']} | anomalies={int(r['anomaly_count'])} | earliest={ts_str} | "
            f"extremal={r['extremal_value_in_window']:.6g} | threshold={r['threshold_value']:.6g} | breach_ratio={br_str}"
        )
    summary = (
        f"Trace anomaly summary (frontend-0/1/2, 2022-03-21 01:30–01:59:59 UTC+8):\n"
        f"- Total anomalous (cmdb_id, trace_name) rows: {total}. Components: {components}.\n"
        f"- Top breaches (by breach_ratio):\n  - " + "\n  - ".join(lines) + "\n\n"
        f"Notes: Most anomalies are duration-related (duration_mean/duration_p95) from frontend pods calling backend services "
        f"(cartservice, adservice, shippingservice, currencyservice). Several entries show very large or infinite breach_ratio "
        f"(threshold reported as 0) indicating a spike above the historical threshold. Anomaly counts per group are small (1–2 samples), "
        f"suggesting short-lived latency spikes. Recommend investigating backend service latency and request paths from frontend pods "
        f"(cartservice/adservice/shippingservice) around the reported earliest timestamps."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                trace_name  anomaly_count earliest_anomaly_timestamp  extremal_value_in_window  threshold_value  breach_ratio
0   frontend-2      trace.to_cartservice-2.duration_mean              1  2022-03-21 01:41:00+08:00                  0.000111         0.000000           inf
1   frontend-2       trace.to_cartservice-2.duration_p95              1  2022-03-21 01:41:00+08:00                  0.000600         0.000000           inf
2   frontend-1         trace.to_adservice-0.duration_p95              2  2022-03-21 01:34:00+08:00                  0.000173         0.000022      7.792793
3   frontend-1      trace.to_cartservice-2.duration_mean              2  2022-03-21 01:31:00+08:00                  0.000571         0.000135      4.232765
4   frontend-0         trace.to_adservice-1.duration_p95              2  2022-03-21 01:43:00+08:00                  0.000104         0.000025      4.219067
5   frontend-1        trace.to_adservice-0.duration_mean              2  2022-03-21 01:34:00+08:00                  0.000061         0.000016      3.922830
6   frontend-1       trace.to_cartservice-2.duration_p95              2  2022-03-21 01:31:00+08:00                  0.002400         0.000677      3.542435
7   frontend-1   trace.to_shippingservice-0.duration_p95              2  2022-03-21 01:39:00+08:00                  0.000287         0.000089      3.228346
8   frontend-2   trace.to_shippingservice-2.duration_p95              2  2022-03-21 01:46:00+08:00                  0.000277         0.000086      3.213457
9   frontend-2   trace.to_currencyservice-0.duration_p95              2  2022-03-21 01:38:00+08:00                  0.000419         0.000136      3.069597
10  frontend-2         trace.to_adservice-0.duration_p95              2  2022-03-21 01:33:00+08:00                  0.000068         0.000026      2.605364
11  frontend-0       trace.to_cartservice-2.duration_p95              2  2022-03-21 01:31:00+08:00                  0.001100         0.000532      2.065728
12  frontend-0  trace.to_shippingservice-0.duration_mean              2  2022-03-21 01:55:00+08:00                  0.000272         0.000133      2.039745
13  frontend-0      trace.to_cartservice-2.duration_mean              2  2022-03-21 01:31:00+08:00                  0.000200         0.000098      2.032520
14  frontend-0       trace.to_cartservice-1.duration_p95              2  2022-03-21 01:46:00+08:00                  0.000500         0.000247      2.020202
15  frontend-2  trace.to_shippingservice-2.duration_mean              2  2022-03-21 01:46:00+08:00                  0.000173         0.000086      2.006961
16  frontend-1  trace.to_shippingservice-1.duration_mean              2  2022-03-21 01:33:00+08:00                  0.000323         0.000161      2.000619
17  frontend-2       trace.to_cartservice-1.duration_p95              1  2022-03-21 01:39:00+08:00                  0.001400         0.000700      2.000000
18  frontend-0      trace.to_cartservice-0.duration_mean              1  2022-03-21 01:35:00+08:00                  0.000182         0.000091      2.000000
19  frontend-2      trace.to_cartservice-1.duration_mean              1  2022-03-21 01:39:00+08:00                  0.000286         0.000143      2.000000```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric and tz if available
try:
    metric
    tz
except NameError:
    metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric types
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Compute global P95 per (cmdb_id, kpi_name) using the full series (rule 9)
p95_df = (
    metric
    .groupby(['cmdb_id','kpi_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)

# Prepare local timestamp (UTC->UTC+8)
metric['ts_local'] = pd.to_datetime(metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Define target components and time window (inclusive)
targets = [
    "cartservice-0","cartservice-1","cartservice-2",
    "adservice-0","adservice-1","adservice-2",
    "shippingservice-0","shippingservice-1","shippingservice-2",
    "currencyservice-0"
]
window_start = pd.Timestamp("2022-03-21 01:30:00", tz=tz)
window_end = pd.Timestamp("2022-03-21 01:59:59", tz=tz)

# Filter metric rows by targets and time window
m_win = metric[
    metric['cmdb_id'].isin(targets) &
    (metric['ts_local'] >= window_start) &
    (metric['ts_local'] <= window_end)
].copy()

# Join with global p95 thresholds (inner join ensures threshold computed)
m_win = m_win.merge(p95_df, on=['cmdb_id','kpi_name'], how='inner')

# Flag anomalies where value > p95_value
m_win['is_anom'] = m_win['value'] > m_win['p95_value']

# Aggregate per (cmdb_id, kpi_name)
grp = m_win.groupby(['cmdb_id','kpi_name'], sort=False)
kp_agg = grp.agg(
    anomaly_count=('is_anom', 'sum'),
    earliest_anom_ts = ('ts_local', lambda s: s[m_win.loc[s.index, 'is_anom']].min() if any(m_win.loc[s.index, 'is_anom']) else pd.NaT),
    max_value_in_window=('value', 'max'),
    p95_value=('p95_value', 'first')
).reset_index()

# Keep only KPIs with at least one anomaly
faulty_kpis = kp_agg[kp_agg['anomaly_count'] > 0].copy()

# Compute breach_ratio = max_value_in_window / p95_value (handle p95==0)
faulty_kpis['top_breach_ratio'] = np.where(
    faulty_kpis['p95_value'] == 0,
    np.where(faulty_kpis['max_value_in_window'] > 0, np.inf, np.nan),
    faulty_kpis['max_value_in_window'] / faulty_kpis['p95_value']
)

# Now aggregate per cmdb_id to produce the requested compact summary
def pick_top_kpi(df):
    # pick row with largest breach ratio; tie-breaker: larger max_value
    df_sorted = df.sort_values(['top_breach_ratio','max_value_in_window'], ascending=[False, False])
    top = df_sorted.iloc[0]
    return pd.Series({
        'num_faulty_kpis': df.shape[0],
        'earliest_anomaly_timestamp': df['earliest_anom_ts'].min(),
        'top_kpi': top['kpi_name'],
        'top_kpi_max_value': top['max_value_in_window'],
        'top_kpi_p95': top['p95_value'],
        'top_kpi_breach_ratio': top['top_breach_ratio']
    })

comp_summary = (
    faulty_kpis
    .groupby('cmdb_id', sort=False)
    .apply(pick_top_kpi)
    .reset_index()
)

# Sort by earliest_anomaly_timestamp ascending and limit to top 20
comp_summary = comp_summary.sort_values('earliest_anomaly_timestamp', ascending=True).head(20)

# Keep columns ordered as requested
final_summary = comp_summary.loc[:, [
    'cmdb_id',
    'num_faulty_kpis',
    'earliest_anomaly_timestamp',
    'top_kpi',
    'top_kpi_max_value',
    'top_kpi_p95',
    'top_kpi_breach_ratio'
]]

# Display compact result (up to 20 rows)
final_summary

```
Out[5]:
```
Summary of anomalies (window 2022-03-21 01:30:00 → 01:59:59 UTC+8), one row per component (compact):

- adservice-2 — num_faulty_kpis: 15; earliest: 2022-03-21 01:30:00+08:00; top_kpi: container.node-5.container_network_receive_MB...; top_max=0.575527; top_p95=0.037824; breach_ratio≈15.216
- cartservice-0 — num_faulty_kpis: 27; earliest: 2022-03-21 01:30:00+08:00; top_kpi: container.node-5.container_network_receive_MB...; top_max=0.305963; top_p95=0.186836; breach_ratio≈1.638
- cartservice-1 — num_faulty_kpis: 29; earliest: 2022-03-21 01:30:00+08:00; top_kpi: container.node-5.container_network_receive_MB...; top_max=0.312114; top_p95=0.188752; breach_ratio≈1.654
- cartservice-2 — num_faulty_kpis: 30; earliest: 2022-03-21 01:30:00+08:00; top_kpi: container.node-5.container_memory_failures.con...; top_max=304.0; top_p95=151.525; breach_ratio≈2.006
- shippingservice-1 — num_faulty_kpis: 31; earliest: 2022-03-21 01:30:00+08:00; top_kpi: mesh.destination.checkoutservice.shippingservi...; top_max=0.000525; top_p95=0.0; breach_ratio=inf (threshold reported 0)
- adservice-0 — num_faulty_kpis: 26; earliest: 2022-03-21 01:31:00+08:00; top_kpi: container.node-5.container_network_receive_MB...; top_max=0.580233; top_p95=0.047205; breach_ratio≈12.292
- shippingservice-0 — num_faulty_kpis: 26; earliest: 2022-03-21 01:31:00+08:00; top_kpi: container.node-5.container_network_receive_MB...; top_max=0.290830; top_p95=0.173207; breach_ratio≈1.679
- shippingservice-2 — num_faulty_kpis: 31; earliest: 2022-03-21 01:31:00+08:00; top_kpi: mesh.destination.checkoutservice.shippingservi...; top_max=0.001050; top_p95=0.0; breach_ratio=inf (threshold reported 0)
- adservice-1 — num_faulty_kpis: 24; earliest: 2022-03-21 01:32:00+08:00; top_kpi: mesh.source.adservice.jaeger-collector.istio_r...; top_max=0.008725; top_p95=0.003400; breach_ratio≈2.566
- currencyservice-0 — num_faulty_kpis: 27; earliest: 2022-03-21 01:36:00+08:00; top_kpi: container.node-5.container_network_receive_MB...; top_max=0.591809; top_p95=0.073419; breach_ratio≈8.061

Key points:
- Earliest anomalies start at 01:30 across multiple services.
- Top breaches are dominated by network_receive_MB metrics (and one memory_failures metric) tied to container.node-5; several components show very large breach ratios (notably adservice-2, adservice-0, currencyservice-0).
- Two entries show infinite breach_ratio because the historical threshold was 0 (indicating a non-zero spike above a zero baseline).
- Recommendation: prioritize investigating node-5 network/memory behavior and the mesh/checkout-related traces for shippingservice calls.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  num_faulty_kpis earliest_anomaly_timestamp                                            top_kpi  top_kpi_max_value  top_kpi_p95  top_kpi_breach_ratio
2        adservice-2               15  2022-03-21 01:30:00+08:00  container.node-5.container_network_receive_MB....           0.575527     0.037824             15.216075
3      cartservice-0               27  2022-03-21 01:30:00+08:00  container.node-5.container_network_receive_MB....           0.305963     0.186836              1.637601
4      cartservice-1               29  2022-03-21 01:30:00+08:00  container.node-5.container_network_receive_MB....           0.312114     0.188752              1.653564
5      cartservice-2               30  2022-03-21 01:30:00+08:00  container.node-5.container_memory_failures.con...         304.000000   151.525000              2.006270
8  shippingservice-1               31  2022-03-21 01:30:00+08:00  mesh.destination.checkoutservice.shippingservi...           0.000525     0.000000                   inf
0        adservice-0               26  2022-03-21 01:31:00+08:00  container.node-5.container_network_receive_MB....           0.580233     0.047205             12.291869
7  shippingservice-0               26  2022-03-21 01:31:00+08:00  container.node-5.container_network_receive_MB....           0.290830     0.173207              1.679085
9  shippingservice-2               31  2022-03-21 01:31:00+08:00  mesh.destination.checkoutservice.shippingservi...           0.001050     0.000000                   inf
1        adservice-1               24  2022-03-21 01:32:00+08:00  mesh.source.adservice.jaeger-collector.istio_r...           0.008725     0.003400              2.566176
6  currencyservice-0               27  2022-03-21 01:36:00+08:00  container.node-5.container_network_receive_MB....           0.591809     0.073419              8.060668```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables where available
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

try:
    p95_df
except NameError:
    # compute global P95 if not present (must be global across full series)
    p95_df = metric.groupby(['cmdb_id','kpi_name'], sort=False)['value'].quantile(0.95).reset_index(name='p95_value')

# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Ensure types and local timestamp column
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['ts_local'] = pd.to_datetime(metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Targets and window (UTC+8)
targets = [
    "cartservice-0","cartservice-1","cartservice-2",
    "adservice-0","adservice-1","adservice-2",
    "shippingservice-0","shippingservice-1","shippingservice-2",
    "currencyservice-0"
]
window_start = pd.Timestamp("2022-03-21 01:30:00", tz=tz)
window_end = pd.Timestamp("2022-03-21 01:59:59", tz=tz)

# Join with global P95 thresholds (global thresholds computed beforehand)
m = metric.merge(p95_df, on=['cmdb_id','kpi_name'], how='inner')

# Filter by target components and time window
m_win = m[
    m['cmdb_id'].isin(targets) &
    (m['ts_local'] >= window_start) &
    (m['ts_local'] <= window_end)
].copy()

# Identify anomalies where value > p95_value
m_win['is_anom'] = m_win['value'] > m_win['p95_value']

# Keep only anomaly rows
anoms = m_win[m_win['is_anom']].copy()
if anoms.empty:
    earliest_anoms = pd.DataFrame(columns=[
        'cmdb_id','earliest_anomaly_timestamp','kpi_name','value_at_earliest','p95_value','breach_ratio'
    ])
else:
    # compute breach ratio safely (handle p95 == 0)
    anoms['breach_ratio'] = np.where(
        anoms['p95_value'] == 0,
        np.where(anoms['value'] > 0, np.inf, np.nan),
        anoms['value'] / anoms['p95_value']
    )

    # For each component find the earliest anomaly timestamp
    earliest_ts = anoms.groupby('cmdb_id', sort=False)['ts_local'].min().reset_index(name='earliest_anomaly_timestamp')

    # Merge to keep only rows at earliest timestamp per component
    anoms = anoms.merge(earliest_ts, on='cmdb_id', how='inner')
    # select rows where ts_local equals earliest timestamp
    mask = anoms['ts_local'] == anoms['earliest_anomaly_timestamp']
    earliest_rows = anoms[mask].copy()

    # If multiple KPIs at same earliest timestamp, pick the one with largest breach_ratio, tie-breaker by max value
    earliest_rows_sorted = earliest_rows.sort_values(['cmdb_id','breach_ratio','value'], ascending=[True, False, False])
    picked = earliest_rows_sorted.groupby('cmdb_id', sort=False).first().reset_index()

    # Prepare final compact columns
    earliest_anoms = picked.loc[:, [
        'cmdb_id',
        'earliest_anomaly_timestamp',
        'kpi_name',
        'value',
        'p95_value',
        'breach_ratio'
    ]].rename(columns={
        'value':'value_at_earliest'
    })

    # Sort by earliest_anomaly_timestamp ascending and limit to 20
    earliest_anoms = earliest_anoms.sort_values('earliest_anomaly_timestamp', ascending=True).head(20).reset_index(drop=True)

# Display compact result
earliest_anoms

```
Out[6]:
```
Summary of earliest anomalies per component (window 2022-03-21 01:30:00 → 01:59:59 UTC+8):

- adservice-2 — earliest: 2022-03-21 01:30:00+08:00. KPI: container.node-5.container_network_transmit_MB...; value=0.593052; p95=0.592727; breach_ratio≈1.001
- cartservice-0 — earliest: 2022-03-21 01:30:00+08:00. KPI: mesh.destination.frontend.cartservice.istio_re...; value=29.5; p95=29.275; breach_ratio≈1.008
- cartservice-1 — earliest: 2022-03-21 01:30:00+08:00. KPI: mesh.source.cartservice.redis-cart.istio_tcp_r...; value=2924.5; p95=2802.1; breach_ratio≈1.044
- cartservice-2 — earliest: 2022-03-21 01:30:00+08:00. KPI: mesh.source.cartservice.redis-cart.istio_tcp_r...; value=2467.5; p95=2463.45; breach_ratio≈1.002
- shippingservice-1 — earliest: 2022-03-21 01:30:00+08:00. KPI: mesh.destination.checkoutservice.shippingservi...; value=0.000525; p95=0.0; breach_ratio=inf (threshold reported 0)
- adservice-0 — earliest: 2022-03-21 01:31:00+08:00. KPI: container.node-5.container_network_receive_MB....; value=0.580233; p95=0.047205; breach_ratio≈12.292
- shippingservice-0 — earliest: 2022-03-21 01:31:00+08:00. KPI: container.node-5.container_cpu_user_seconds; value=0.015; p95=0.01; breach_ratio=1.5
- shippingservice-2 — earliest: 2022-03-21 01:31:00+08:00. KPI: container.node-5.container_cpu_usage_seconds; value=0.016726; p95=0.016162; breach_ratio≈1.035
- adservice-1 — earliest: 2022-03-21 01:32:00+08:00. KPI: mesh.source.adservice.jaeger-collector.istio_r...; value=0.004075; p95=0.0034; breach_ratio≈1.199
- currencyservice-0 — earliest: 2022-03-21 01:36:00+08:00. KPI: container.node-5.container_memory_failures.con...; value=1142.0; p95=1045.7; breach_ratio≈1.092

Notes: All listed components had at least one metric value exceeding their global P95 within the window. shippingservice-1 shows an infinite breach_ratio because the historical P95 was zero while the observed value was non‑zero. The most pronounced early breaches (by ratio) include adservice-0 (very large network_receive breach) and several mesh/network KPIs across cartservice and adservice.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_anomaly_timestamp                                           kpi_name  value_at_earliest    p95_value  breach_ratio
0        adservice-2  2022-03-21 01:30:00+08:00  container.node-5.container_network_transmit_MB...           0.593052     0.592727      1.000550
1      cartservice-0  2022-03-21 01:30:00+08:00  mesh.destination.frontend.cartservice.istio_re...          29.500000    29.275000      1.007686
2      cartservice-1  2022-03-21 01:30:00+08:00  mesh.source.cartservice.redis-cart.istio_tcp_r...        2924.500000  2802.100000      1.043682
3      cartservice-2  2022-03-21 01:30:00+08:00  mesh.source.cartservice.redis-cart.istio_tcp_r...        2467.500000  2463.450000      1.001644
4  shippingservice-1  2022-03-21 01:30:00+08:00  mesh.destination.checkoutservice.shippingservi...           0.000525     0.000000           inf
5        adservice-0  2022-03-21 01:31:00+08:00  container.node-5.container_network_receive_MB....           0.580233     0.047205     12.291869
6  shippingservice-0  2022-03-21 01:31:00+08:00        container.node-5.container_cpu_user_seconds           0.015000     0.010000      1.500000
7  shippingservice-2  2022-03-21 01:31:00+08:00       container.node-5.container_cpu_usage_seconds           0.016726     0.016162      1.034900
8        adservice-1  2022-03-21 01:32:00+08:00  mesh.source.adservice.jaeger-collector.istio_r...           0.004075     0.003400      1.198529
9  currencyservice-0  2022-03-21 01:36:00+08:00  container.node-5.container_memory_failures.con...        1142.000000  1045.700000      1.092091```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# reuse existing error_logs and tz where possible
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp numeric and create localized timestamp in UTC+8
error_logs['timestamp'] = pd.to_numeric(error_logs['timestamp'], errors='coerce').astype('Int64')
error_logs['ts_local'] = pd.to_datetime(error_logs['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Filter to frontends
frontends = ["frontend-0","frontend-1","frontend-2"]
errs_f = error_logs[error_logs['cmdb_id'].isin(frontends)].copy()

# 1) Per-frontend summary: total message count, earliest and latest timestamp
frontend_summary = (
    errs_f
    .groupby('cmdb_id', sort=False)
    .agg(
        total_messages = ('message','size'),
        earliest_ts = ('ts_local','min'),
        latest_ts = ('ts_local','max')
    )
    .reset_index()
)

# 2) Top up to 10 unique message texts with counts
top_n = 10
msg_counts = errs_f['message'].value_counts().head(top_n).rename_axis('message').reset_index(name='count')

# 3) For each unique message add substring presence flags and first example row
substrs = ['cartservice','adservice','shippingservice','currencyservice','node-5','failed to retrieve','timeout','connection refused']

# Build example rows (first occurrence) for messages present in errs_f
examples = (
    errs_f.sort_values('ts_local')
    .groupby('message', sort=False)
    .agg(example_ts = ('ts_local','first'),
         example_cmdb = ('cmdb_id','first'),
         example_msg = ('message','first'))
    .reset_index()
)

# Merge counts with examples for top messages
top_messages = msg_counts.merge(examples, on='message', how='left')

# Add substring presence columns (present/absent, case-insensitive)
for s in substrs:
    col = f"has_{s.replace(' ','_')}"
    top_messages[col] = top_messages['message'].str.contains(s, case=False, na=False).map({True:'present', False:'absent'})

# Keep only requested columns and keep compact (limit rows <=20)
cols = ['message','count','example_ts','example_cmdb'] + [f"has_{s.replace(' ','_')}" for s in substrs]
top_messages = top_messages.loc[:, cols].head(20)

# Final compact output variables
frontend_summary, top_messages

```
Out[7]:
```
Summary of error_logs for frontend-0/1/2:

1) Per-frontend totals and time range
- frontend-0: 233 messages; earliest 2022-03-21 01:30:04+08:00; latest 2022-03-21 01:59:52+08:00
- frontend-1: 155 messages; earliest 2022-03-21 01:30:04+08:00; latest 2022-03-21 01:59:56+08:00
- frontend-2: 155 messages; earliest 2022-03-21 01:30:06+08:00; latest 2022-03-21 01:59:59+08:00

2) Top message texts (up to 10)
- "severity: warning, message: failed to retrieve..." — count: 543  
  Example first occurrence: 2022-03-21 01:30:04+08:00 (frontend-0)

3) Substring presence for each unique message (present/absent)
- For the top message above:
  - cartservice: absent
  - adservice: absent
  - shippingservice: absent
  - currencyservice: absent
  - node-5: absent
  - failed to retrieve: present
  - timeout: absent
  - connection refused: absent

Interpretation: Almost all error log entries from these frontends are the same warning "failed to retrieve..." (543 occurrences) starting at ~01:30:04 and appearing across frontend-0/1/2. The message text does not mention backend service names or node-5, but explicitly contains "failed to retrieve", suggesting a retrieval/fetch failure originating at frontends; investigate frontend fetch logic, upstream calls, or network paths.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_messages               earliest_ts                 latest_ts
0  frontend-0             233 2022-03-21 01:30:04+08:00 2022-03-21 01:59:52+08:00
1  frontend-1             155 2022-03-21 01:30:04+08:00 2022-03-21 01:59:56+08:00
2  frontend-2             155 2022-03-21 01:30:06+08:00 2022-03-21 01:59:59+08:00,                                              message  count                example_ts example_cmdb has_cartservice has_adservice has_shippingservice has_currencyservice has_node-5 has_failed_to_retrieve has_timeout has_connection_refused
0  severity: warning, message: failed to retrieve...    543 2022-03-21 01:30:04+08:00   frontend-0          absent        absent              absent              absent     absent                present      absent                 absent)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
import re

# reuse existing trace, thresholds, tz if present; otherwise compute required globals
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

try:
    thresholds
except NameError:
    # compute global P95 and P5 across full trace series (per cmdb_id, trace_name)
    trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce').astype('Int64')
    trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
    grp_all = trace.groupby(['cmdb_id','trace_name'], sort=False)
    p95 = grp_all['value'].quantile(0.95).reset_index(name='p95_value')
    p5 = grp_all['value'].quantile(0.05).reset_index(name='p5_value')
    thresholds = p95.merge(p5, on=['cmdb_id','trace_name'], how='left')
    thresholds['is_row_count'] = thresholds['trace_name'].str.contains('row_count', case=False, na=False)
    thresholds['threshold_value'] = np.where(thresholds['is_row_count'], thresholds['p5_value'], thresholds['p95_value'])
    thresholds = thresholds[['cmdb_id','trace_name','threshold_value','is_row_count']]

# Ensure tz
tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric and local timestamp on trace
trace['timestamp'] = pd.to_numeric(trace['timestamp'], errors='coerce').astype('Int64')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['ts_local'] = pd.to_datetime(trace['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Filter criteria
frontends = ["frontend-0","frontend-1","frontend-2"]
patterns = ['to_cartservice','to_adservice','to_shippingservice','to_currencyservice']
pattern_regex = '|'.join(patterns)

window_start = pd.Timestamp("2022-03-21 01:30:00", tz=tz)
window_end = pd.Timestamp("2022-03-21 01:59:59", tz=tz)

# Global thresholds are already computed in `thresholds` (per RULE 9)
# Filter trace rows by frontend cmdb_id, trace_name patterns, and time window
t_filtered = trace[
    trace['cmdb_id'].isin(frontends) &
    trace['trace_name'].str.contains(pattern_regex, case=False, na=False) &
    (trace['ts_local'] >= window_start) &
    (trace['ts_local'] <= window_end)
].copy()

# Join with global thresholds (on cmdb_id and trace_name)
t_join = t_filtered.merge(thresholds, on=['cmdb_id','trace_name'], how='inner')

# Determine anomaly condition per row
t_join['is_row_count'] = t_join['is_row_count'].astype(bool)
t_join['is_anom'] = np.where(t_join['is_row_count'], t_join['value'] < t_join['threshold_value'], t_join['value'] > t_join['threshold_value'])

# Extract target_service from trace_name, e.g., trace.to_cartservice-2.duration_mean -> cartservice-2
def extract_target(trace_name):
    m = re.search(r'to_([^.]+)', trace_name)
    return m.group(1) if m else ''

t_join['target_service'] = t_join['trace_name'].apply(extract_target)

# Determine anomaly_type from trace_name (duration_mean/duration_p95/error_rate/row_count)
def anomaly_type_from_name(name):
    for t in ['duration_mean','duration_p95','error_rate','row_count']:
        if t in name:
            return t
    return 'other'

t_join['anomaly_type'] = t_join['trace_name'].apply(anomaly_type_from_name)

# Aggregate per (trace_name, target_service, anomaly_type)
grp = t_join.groupby(['trace_name','target_service','anomaly_type','is_row_count'], sort=False)

agg = grp.agg(
    anomaly_count = ('is_anom', 'sum'),
    earliest_anom_ts = ('ts_local', lambda s: s[t_join.loc[s.index,'is_anom']].min() if any(t_join.loc[s.index,'is_anom']) else pd.NaT),
    max_value_in_window = ('value', 'max'),
    min_value_in_window = ('value', 'min'),
    threshold_value = ('threshold_value', 'first')
).reset_index()

# Compute extremal_value_in_window and breach_ratio
agg['extremal_value_in_window'] = np.where(agg['is_row_count'], agg['min_value_in_window'], agg['max_value_in_window'])

def compute_breach(row):
    thr = row['threshold_value']
    ext = row['extremal_value_in_window']
    if pd.isna(thr) or pd.isna(ext):
        return np.nan
    if row['is_row_count']:
        if ext == 0:
            return np.inf if thr > 0 else np.nan
        return thr / ext
    else:
        if thr == 0:
            return np.inf if ext > 0 else np.nan
        return ext / thr

agg['breach_ratio'] = agg.apply(compute_breach, axis=1)

# Keep only groups with at least one anomaly
summary = agg[agg['anomaly_count'] > 0].copy()

# Select requested columns and sort by earliest_anom_ts ascending, limit to 20 rows
if not summary.empty:
    result = summary.loc[:, [
        'trace_name','target_service','anomaly_type','anomaly_count','earliest_anom_ts',
        'extremal_value_in_window','threshold_value','breach_ratio'
    ]].sort_values('earliest_anom_ts', ascending=True).head(20).reset_index(drop=True)
else:
    result = pd.DataFrame(columns=[
        'trace_name','target_service','anomaly_type','anomaly_count','earliest_anom_ts',
        'extremal_value_in_window','threshold_value','breach_ratio'
    ])

# Display compact result
result

```
Out[8]:
```
Summary of trace anomalies for frontend-0/1/2 calling backend services (window 2022-03-21 01:30:00 → 01:59:59 UTC+8):

- Scope: traces containing to_cartservice / to_adservice / to_shippingservice / to_currencyservice from frontend-0/1/2. Global thresholds (P95 or P5) were used to detect anomalies.

High-level findings (one row per trace_name / target service, showing anomaly_type, anomaly_count, earliest timestamp, extremal value, threshold, breach_ratio):

- trace.to_adservice-2.duration_mean — adservice-2; duration_mean; 6 anomalies; earliest 01:30; extremal 0.000017 vs p95 0.000015; breach_ratio ~1.17.
- trace.to_adservice-2.duration_p95 — adservice-2; duration_p95; 6 anomalies; earliest 01:30; extremal 0.000032 vs p95 0.000022; breach_ratio ~1.48.
- trace.to_currencyservice-2.duration_p95 — currencyservice-2; duration_p95; 5 anomalies; earliest 01:30; breach_ratio ~1.21.
- trace.to_currencyservice-1.duration_mean — currencyservice-1; duration_mean; 6 anomalies; earliest 01:30; breach_ratio ~1.49.
- trace.to_adservice-0.row_count — adservice-0; row_count (low); 3 anomalies; earliest 01:31; extremal 3 (min) vs p5 6; breach_ratio = 2.00 (threshold/min).
- trace.to_adservice-1.duration_mean — adservice-1; duration_mean; 3 anomalies; earliest 01:31; breach_ratio = 2.00.
- trace.to_currencyservice-2.duration_mean — currencyservice-2; duration_mean; 4 anomalies; earliest 01:31; breach_ratio ~1.11.
- trace.to_currencyservice-1.duration_p95 — currencyservice-1; duration_p95; 4 anomalies; earliest 01:31; breach_ratio ~1.14.
- trace.to_cartservice-2.duration_mean — cartservice-2; duration_mean; 5 anomalies; earliest 01:31; extremal 0.000571 vs p95 0.000098; breach_ratio ~5.80 (largest mean-duration spike).
- trace.to_cartservice-2.duration_p95 — cartservice-2; duration_p95; 5 anomalies; earliest 01:31; extremal 0.002400 vs p95 0.000532; breach_ratio ~4.51.
- trace.to_adservice-0.duration_mean — adservice-0; duration_mean; 4 anomalies; earliest 01:33; breach_ratio ~3.59.
- trace.to_shippingservice-1.duration_p95 — shippingservice-1; duration_p95; 6 anomalies; earliest 01:33; breach_ratio ~2.15.
- trace.to_shippingservice-1.duration_mean — shippingservice-1; duration_mean; 6 anomalies; earliest 01:33; breach_ratio ~3.38.
- Several row_count (throughput) reductions were detected:
  - currencyservice-2.row_count, currencyservice-1.row_count, currencyservice-0.row_count — each 6 anomalies, earliest 01:33; extremal ~13 vs thresholds ~24–25; breach_ratio ~1.9–1.96 (meaning observed row_count about half of baseline).
  - cartservice-2.row_count — 2 anomalies, earliest 01:33; extremal 5 vs p5 10; breach_ratio = 2.00.
  - cartservice-0.row_count — 1 anomaly, earliest 01:33; extremal 6 vs p5 10; breach_ratio ~1.67.
  - adservice-2.row_count — 3 anomalies, earliest 01:33; extremal 3 vs p5 6; breach_ratio = 2.00.

Interpretation and recommended focus:
- Strong latency spikes from frontends to cartservice (cartservice-2) and adservice (adservice-0/2) — cartservice-2 shows the largest duration spikes (breach_ratio ~5.8 and ~4.5 for mean and p95).
- Concurrent reductions in row_count (throughput) to currencyservice and cartservice indicate degraded request throughput around ~01:33.
- Shippingservice shows elevated durations as well (breach_ratio up to ~3.38).
- Timeline: many anomalies begin at 01:30–01:33 UTC+8 (earliest detections).
- Actionable next steps: investigate backend pod health and network/path for cartservice and adservice (check pod logs, CPU/IO/network on target pods), and examine currencyservice throughput issues (queueing, downstream calls, or resource saturation). Also correlate with node-5 metrics (previously flagged) since multiple KPIs referenced container.node-5 in metric anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

trace_name     target_service   anomaly_type  anomaly_count          earliest_anom_ts  extremal_value_in_window  threshold_value  breach_ratio
0         trace.to_adservice-2.duration_mean        adservice-2  duration_mean              6 2022-03-21 01:30:00+08:00                  0.000017         0.000015      1.168385
1          trace.to_adservice-2.duration_p95        adservice-2   duration_p95              6 2022-03-21 01:30:00+08:00                  0.000032         0.000022      1.484919
2    trace.to_currencyservice-2.duration_p95  currencyservice-2   duration_p95              5 2022-03-21 01:30:00+08:00                  0.000146         0.000121      1.211116
3   trace.to_currencyservice-1.duration_mean  currencyservice-1  duration_mean              6 2022-03-21 01:30:00+08:00                  0.000158         0.000106      1.489161
4             trace.to_adservice-0.row_count        adservice-0      row_count              3 2022-03-21 01:31:00+08:00                  3.000000         6.000000      2.000000
5         trace.to_adservice-1.duration_mean        adservice-1  duration_mean              3 2022-03-21 01:31:00+08:00                  0.000030         0.000015      2.000000
6   trace.to_currencyservice-2.duration_mean  currencyservice-2  duration_mean              4 2022-03-21 01:31:00+08:00                  0.000125         0.000113      1.105705
7    trace.to_currencyservice-1.duration_p95  currencyservice-1   duration_p95              4 2022-03-21 01:31:00+08:00                  0.000139         0.000122      1.139344
8       trace.to_cartservice-2.duration_mean      cartservice-2  duration_mean              5 2022-03-21 01:31:00+08:00                  0.000571         0.000098      5.802846
9        trace.to_cartservice-2.duration_p95      cartservice-2   duration_p95              5 2022-03-21 01:31:00+08:00                  0.002400         0.000532      4.507042
10        trace.to_adservice-0.duration_mean        adservice-0  duration_mean              4 2022-03-21 01:33:00+08:00                  0.000061         0.000017      3.588235
11   trace.to_shippingservice-1.duration_p95  shippingservice-1   duration_p95              6 2022-03-21 01:33:00+08:00                  0.000323         0.000150      2.154770
12  trace.to_shippingservice-1.duration_mean  shippingservice-1  duration_mean              6 2022-03-21 01:33:00+08:00                  0.000323         0.000096      3.380429
13      trace.to_currencyservice-2.row_count  currencyservice-2      row_count              6 2022-03-21 01:33:00+08:00                 13.000000        24.900000      1.915385
14      trace.to_currencyservice-1.row_count  currencyservice-1      row_count              6 2022-03-21 01:33:00+08:00                 13.000000        25.450000      1.957692
15      trace.to_currencyservice-0.row_count  currencyservice-0      row_count              6 2022-03-21 01:33:00+08:00                 13.000000        24.450000      1.880769
16          trace.to_cartservice-2.row_count      cartservice-2      row_count              2 2022-03-21 01:33:00+08:00                  5.000000        10.000000      2.000000
17          trace.to_cartservice-0.row_count      cartservice-0      row_count              1 2022-03-21 01:33:00+08:00                  6.000000        10.000000      1.666667
18            trace.to_adservice-2.row_count        adservice-2      row_count              3 2022-03-21 01:33:00+08:00                  3.000000         6.000000      2.000000
19         trace.to_adservice-0.duration_p95        adservice-0   duration_p95              6 2022-03-21 01:33:00+08:00                  0.000173         0.000032      5.389408```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric and tz if available
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric types and local timestamp (UTC+8)
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['ts_local'] = pd.to_datetime(metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# Rule 9: compute global P95 for every (cmdb_id, kpi_name) across the full series BEFORE time filtering
p95_all = (
    metric
    .groupby(['cmdb_id','kpi_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index(name='p95_value')
)

# KPI name substrings to match (focus on network/error/retransmit/packet related KPIs)
substrs = [
    'network_receive_err','network_transmit_err','network_retransmit',
    'network_receive_packet','network_transmit_packet',
    'container_network_receive_err','container_network_transmit_err',
    'container_network_retransmit','container_network_receive_packet','container_network_transmit_packet'
]
regex = '|'.join(substrs)

# Select node-5 KPIs of interest based on p95_all (ensures thresholds computed globally)
node5_kpis = p95_all[
    (p95_all['cmdb_id'] == 'node-5') &
    (p95_all['kpi_name'].str.contains(regex, case=False, na=False))
].copy()

# If no matching KPIs, return empty DataFrame with expected columns
if node5_kpis.empty:
    result_node5 = pd.DataFrame(columns=[
        'kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','breach_ratio'
    ])
else:
    # Filter metric rows for node-5 and matching KPI names, then restrict to time window
    kpi_list = node5_kpis['kpi_name'].unique().tolist()
    window_start = pd.Timestamp("2022-03-21 01:30:00", tz=tz)
    window_end = pd.Timestamp("2022-03-21 01:59:59", tz=tz)

    m_node5 = metric[
        (metric['cmdb_id'] == 'node-5') &
        (metric['kpi_name'].isin(kpi_list)) &
        (metric['ts_local'] >= window_start) &
        (metric['ts_local'] <= window_end)
    ].copy()

    # Merge global p95 values
    m_node5 = m_node5.merge(node5_kpis[['cmdb_id','kpi_name','p95_value']],
                            on=['cmdb_id','kpi_name'], how='left')

    # Determine anomalies (value > p95_value)
    m_node5['is_anom'] = m_node5['value'] > m_node5['p95_value']

    # Aggregate per kpi_name
    grp = m_node5.groupby('kpi_name', sort=False)
    agg = grp.agg(
        anomaly_count = ('is_anom', 'sum'),
        earliest_anomaly_timestamp = ('ts_local', lambda s: s[m_node5.loc[s.index,'is_anom']].min() if any(m_node5.loc[s.index,'is_anom']) else pd.NaT),
        max_value_in_window = ('value', 'max'),
        p95_value = ('p95_value', 'first')
    ).reset_index()

    # Compute breach_ratio with handling p95==0
    def compute_breach(maxv, p95v):
        if pd.isna(p95v) or pd.isna(maxv):
            return np.nan
        if p95v == 0:
            return np.inf if maxv > 0 else np.nan
        return maxv / p95v

    agg['breach_ratio'] = agg.apply(lambda r: compute_breach(r['max_value_in_window'], r['p95_value']), axis=1)

    # Keep only KPIs with at least one anomaly (or p95==0 and max>0 as requested)
    cond_inf = (agg['p95_value'] == 0) & (agg['max_value_in_window'] > 0)
    agg_filtered = agg[(agg['anomaly_count'] > 0) | cond_inf].copy()

    # Sort by breach_ratio descending (show most severe first) and limit to 20 rows
    result_node5 = agg_filtered.sort_values('breach_ratio', ascending=False).loc[:, [
        'kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','p95_value','breach_ratio'
    ]].head(20).reset_index(drop=True)

# Display compact result variable
result_node5

```
Out[9]:
```
```python
summary = (
    "Result: No anomalies found for the requested node-5 network/error KPIs in the specified window.\n\n"
    "Details: When filtering metric KPIs for cmdb_id='node-5' and kpi_name containing any of the requested substrings "
    "('network_receive_err','network_transmit_err','network_retransmit','network_receive_packet','network_transmit_packet', "
    "'container_network_receive_err','container_network_transmit_err','container_network_retransmit', "
    "'container_network_receive_packet','container_network_transmit_packet'), no matching node-5 KPIs were found in the dataset (global P95 table was empty for these patterns). "
    "Therefore there are no anomaly points (value > P95) to report in 2022-03-21 01:30:00 → 01:59:59 UTC+8.\n\n"
    "Recommended next steps: broaden the KPI name filter (use shorter substrings like 'network' or inspect available node-5 kpi_name list) "
    "or check related KPIs (e.g., container.node-5.* network/memory metrics) to detect potential network-related issues on node-5."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [kpi_name, anomaly_count, earliest_anomaly_timestamp, max_value_in_window, p95_value, breach_ratio]
Index: []```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric and tz if present
try:
    metric
    tz
except NameError:
    metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric types and local timestamp (UTC+8)
metric['timestamp'] = pd.to_numeric(metric['timestamp'], errors='coerce').astype('Int64')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['ts_local'] = pd.to_datetime(metric['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz)

# RULE 9: compute global P95 per (cmdb_id, kpi_name) across full series BEFORE time filtering
p95_all = metric.groupby(['cmdb_id','kpi_name'], sort=False)['value'].quantile(0.95).reset_index(name='p95_value')

# Find kpi_names containing 'container.node-5' and any of the requested substrings
substrs = ['network','memory_failures','memory','io','disk']
mask_name = metric['kpi_name'].str.contains('container.node-5', case=False, na=False) & \
            metric['kpi_name'].str.contains('|'.join(substrs), case=False, na=False)
kpi_names = metric.loc[mask_name, 'kpi_name'].unique().tolist()

# Time window (UTC+8)
window_start = pd.Timestamp("2022-03-21 01:30:00", tz=tz)
window_end = pd.Timestamp("2022-03-21 01:59:59", tz=tz)

if not kpi_names:
    result_kpis = pd.DataFrame(columns=[
        'kpi_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','p95_value','breach_ratio','pod_of_max'
    ])
else:
    # Filter rows for these kpi_names within time window
    m_sel = metric[
        metric['kpi_name'].isin(kpi_names) &
        (metric['ts_local'] >= window_start) &
        (metric['ts_local'] <= window_end)
    ].copy()

    # Merge with global p95 per (cmdb_id,kpi_name)
    m_sel = m_sel.merge(p95_all, on=['cmdb_id','kpi_name'], how='left')

    # Determine anomaly flag per row (value > p95)
    m_sel['is_anom'] = m_sel['value'] > m_sel['p95_value']

    # Aggregate per kpi_name:
    # - anomaly_count: sum of is_anom across all pods
    # - earliest_anomaly_timestamp: min ts_local among rows where is_anom
    # - max_value_in_window: max value across window (all pods)
    agg_counts = m_sel.groupby('kpi_name', sort=False).agg(
        anomaly_count = ('is_anom', 'sum'),
        earliest_anomaly_timestamp = ('ts_local', lambda s: s[m_sel.loc[s.index,'is_anom']].min() if any(m_sel.loc[s.index,'is_anom']) else pd.NaT),
        max_value_in_window = ('value', 'max')
    ).reset_index()

    # Find pod (cmdb_id) that has the max_value_in_window per kpi_name
    idx_max = m_sel.groupby('kpi_name')['value'].idxmax().dropna().astype(int)
    pod_of_max = m_sel.loc[idx_max, ['kpi_name','cmdb_id','p95_value','value']].rename(columns={
        'cmdb_id':'pod_of_max',
        'p95_value':'p95_of_pod',
        'value':'value_at_pod_max'
    }).set_index('kpi_name')

    # Merge pod info
    agg = agg_counts.set_index('kpi_name').join(pod_of_max, how='left').reset_index()

    # Use p95_value for the pod_of_max as p95_value to report
    agg['p95_value'] = agg['p95_of_pod']

    # Compute breach_ratio = max_value_in_window / p95_value, handle p95==0
    def breach(maxv, p95v):
        if pd.isna(maxv) or pd.isna(p95v):
            return np.nan
        if p95v == 0:
            return np.inf if maxv > 0 else np.nan
        return maxv / p95v

    agg['breach_ratio'] = agg.apply(lambda r: breach(r['max_value_in_window'], r['p95_value']), axis=1)

    # Keep only KPI rows where anomaly_count>0 or p95==0 and max>0 (per instruction)
    cond_inf = (agg['p95_value'] == 0) & (agg['max_value_in_window'] > 0)
    result_kpis = agg[(agg['anomaly_count'] > 0) | cond_inf].copy()

    # Select and order columns, sort by earliest_anomaly_timestamp ascending, limit to 20 rows
    result_kpis = result_kpis.loc[:, [
        'kpi_name','anomaly_count','earliest_anomaly_timestamp',
        'max_value_in_window','p95_value','breach_ratio','pod_of_max'
    ]].sort_values('earliest_anomaly_timestamp', ascending=True).head(20).reset_index(drop=True)

# Display compact result
result_kpis

```
Out[10]:
```
Summary (node-5 container.* KPIs with anomalies, window 2022-03-21 01:30–01:59:59 UTC+8):

Overview
- Multiple container.node-5.* KPIs show anomalies starting at ~01:30 (UTC+8), affecting network, memory, I/O and CPU-throttle related metrics across several pods.
- Results returned (one row per kpi_name): anomaly_count, earliest anomaly timestamp, max value in window, global P95, breach_ratio (max / p95), and the pod that exhibited the max value.

Key rows (compact):
- container.node-5.container_cpu_cfs_periods — anomaly_count 58; earliest 01:30; max=198.0; p95=178.775; breach_ratio≈1.11; pod_of_max=productcatalogservice-0.
- container.node-5.container_memory_failures... — anomaly_count 51; earliest 01:30; max=77931.5; p95=20113.87; breach_ratio≈3.87; pod_of_max=adservice-0.
- container.node-5.container_memory_failures.high... — same counts/values as above (same pod_of_max=adservice-0).
- container.node-5.container_memory_rss — anomaly_count 47; earliest 01:30; max=147,152,900; p95=147,152,900; breach_ratio=1.00; pod_of_max=adservice-0.
- container.node-5.container_network_receive_MB... — anomaly_count 70; earliest 01:30; max≈0.8798; p95≈0.4363; breach_ratio≈2.02; pod_of_max=productcatalogservice-1.
- container.node-5.container_network_receive_packets... — anomaly_count 69; earliest 01:30; max=2682.0; p95=2397.65; breach_ratio≈1.12; pod_of_max=productcatalogservice-2.
- container.node-5.container_network_transmit_MB... — anomaly_count 70; earliest 01:30; max≈4.3989; p95≈4.3260; breach_ratio≈1.02; pod_of_max=frontend-0.
- container.node-5.container_network_transmit_packets... — anomaly_count 70; earliest 01:30; max=2619.5; p95=2322.075; breach_ratio≈1.13; pod_of_max=productcatalogservice-2.
- container.node-5.container_memory_usage_MB — anomaly_count 46; earliest 01:31; max≈241.0664; p95=241.0664; breach_ratio=1.00; pod_of_max=adservice-0.
- container.node-5.container_memory_working_set_MB — anomaly_count 46; earliest 01:31; max≈222.7930; p95=222.7930; breach_ratio=1.00; pod_of_max=adservice-0.
- container.node-5.container_cpu_cfs_throttled_periods — anomaly_count 4; earliest 01:34; max=24.0; p95=5.5; breach_ratio≈4.36; pod_of_max=adservice-0.

Interpretation / action items
- Memory failure signals are the most severe by ratio (container_memory_failures metrics, breach_ratio ≈ 3.87) and the max values occur on pod adservice-0 — investigate adservice-0 memory faults and allocation/failures.
- Several productcatalogservice pods show elevated network receive/transmit packet and MB rates (productcatalogservice-1/2, productcatalogservice-0) — investigate network traffic or load balancing to those pods.
- frontend-0 shows the highest container_network_transmit_MB peak for node-5 metrics — review frontend network egress at the same times.
- CPU throttling (container_cpu_cfs_throttled_periods) spikes on adservice-0 at ~01:34 (breach_ratio ~4.36) — check CPU limits, throttling, and contention for that pod.
- Many anomalies begin at ~01:30, pointing to an event/window at that time; correlate with trace and error logs (frontends reported "failed to retrieve" warnings) and node-5 metrics.

If you want, I can:
- show the exact pod-level rows (per-kpi per-pod) for the top anomalies, or
- expand the KPI list or sort by breach_ratio instead of earliest timestamp.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window     p95_value  breach_ratio               pod_of_max
0          container.node-5.container_cpu_cfs_periods             58  2022-03-21 01:30:00+08:00         1.980000e+02  1.787750e+02      1.107537  productcatalogservice-0
1   container.node-5.container_memory_failures.con...             51  2022-03-21 01:30:00+08:00         7.793150e+04  2.011387e+04      3.874514              adservice-0
2   container.node-5.container_memory_failures.hie...             51  2022-03-21 01:30:00+08:00         7.793150e+04  2.011387e+04      3.874514              adservice-0
3               container.node-5.container_memory_rss             47  2022-03-21 01:30:00+08:00         1.471529e+08  1.471529e+08      1.000000              adservice-0
4   container.node-5.container_network_receive_MB....             70  2022-03-21 01:30:00+08:00         8.797793e-01  4.363169e-01      2.016377  productcatalogservice-1
5   container.node-5.container_network_receive_pac...             69  2022-03-21 01:30:00+08:00         2.682000e+03  2.397650e+03      1.118595  productcatalogservice-2
6   container.node-5.container_network_transmit_MB...             70  2022-03-21 01:30:00+08:00         4.398902e+00  4.325988e+00      1.016855               frontend-0
7   container.node-5.container_network_transmit_pa...             70  2022-03-21 01:30:00+08:00         2.619500e+03  2.322075e+03      1.128086  productcatalogservice-2
8          container.node-5.container_memory_usage_MB             46  2022-03-21 01:31:00+08:00         2.410664e+02  2.410664e+02      1.000000              adservice-0
9    container.node-5.container_memory_working_set_MB             46  2022-03-21 01:31:00+08:00         2.227930e+02  2.227930e+02      1.000000              adservice-0
10  container.node-5.container_cpu_cfs_throttled_p...              4  2022-03-21 01:34:00+08:00         2.400000e+01  5.500000e+00      4.363636              adservice-0```
```